# 12b — Cífka exact-code validation with GPT-2 + scalable replication

This notebook separates two jobs that should not be conflated.

## A. Exact released-code audit
Run the released `cifkao/context-probing` `run_probing()` **unchanged** with GPT-2, one of the scorer families used by Cífka & Liutkus. The exact path is run on CPU because the released implementation materializes the full vocabulary distribution for every sliding context and is not feasible on an 8 GB GPU at \(k=256\).

The exact audit directly extracts:

\[
L_{256},\qquad (L_8,L_{256}),\qquad
(L_8,L_{16},L_{32},L_{64},L_{128},L_{256})
\]

from the returned `xent` tensor.

## B. Scaled GPT-2 attribution experiment
After exact numerical equivalence is established, run an xent-only implementation of the same Cífka windowing/diagonalization calculation on a larger matched GPT/Claude subset and compare:

- `L256` (1D)
- `Endpoints` (2D)
- `SparseLog6` (6D)
- `DenseCLP` (249D)

This gives a defensible statement:

> We verified the measurement against the unmodified released Cífka–Liutkus implementation using GPT-2, then used a numerically equivalent xent-only implementation to scale the same scorer and dataset comparison.


In [1]:
import sys, subprocess, importlib.util

if importlib.util.find_spec("context_probing") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-deps",
        "git+https://github.com/cifkao/context-probing.git",
    ])

import context_probing
print("context_probing:", context_probing.__file__)


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


context_probing: /home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/context_probing/__init__.py


In [2]:
from pathlib import Path
import hashlib, inspect, time, gc

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

from context_probing import run_probing
import context_probing.core as cp_core
from context_probing.utils import get_windows, columns_to_diagonals

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "experiments" else cwd
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RAW_PATH = DATA / "raw" / "ghostbuster_detector_pool.parquet"
assert RAW_PATH.exists(), RAW_PATH

MODEL_ID = "gpt2"
MAX_CONTEXT = 256

# Exact code: minimum useful length = 257 tokens, giving one target with 256-token history.
EXACT_INPUT_TOKENS = 257
N_EXACT_DOCS = 2

# Scaled replication. Increase later if runtime is acceptable.
N_PAIRS_SCALED = 150
SCALED_INPUT_TOKENS = 384
SUBSET_SEED = 1201

EXACT_CACHE = RESULTS / "cifka_exact_gpt2_audit.parquet"
SCALED_CACHE = RESULTS / "cifka_dense_clp_gpt2.parquet"

print("Exact scorer:", MODEL_ID)


Exact scorer: gpt2


## Record the released code actually executed


In [3]:
core_path = Path(inspect.getsourcefile(cp_core)).resolve()
core_sha256 = hashlib.sha256(core_path.read_bytes()).hexdigest()

print("core.py:", core_path)
print("SHA256:", core_sha256)
print("\nrun_probing source begins:\n")
print("\n".join(inspect.getsource(run_probing).splitlines()[:28]))


core.py: /home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/context_probing/core.py
SHA256: 02ad2cce22bd406d78c19d14311a23e701e4e510e267c7fd4d97e56f47eb674d

run_probing source begins:

@torch.inference_mode()
def run_probing(
    model: Callable[..., Union[ModelOutput, torch.Tensor]],
    inputs: Dict[str, Any],
    window_len: Optional[int] = None,
    metrics: Optional[List[str]] = None,
    tokenizer: Optional[PreTrainedTokenizerBase] = None,
    eos_id: Optional[int] = None,
    unigram_logprobs: Optional[torch.Tensor] = None,
    batch_size: int = 8,
) -> Dict[str, torch.Tensor]:
    """Run context length probing with the given model and inputs.

    Args:
        model: A pretrained PyTorch Hugging Face Transformers causal language model, or a callable
            accepting as parameters PyTorch tensors corresponding to a batch of values from
            `inputs` and returning a either a `ModelOutput` with a `logits` attribute or a logits
           

## Deterministic matched source pool


In [4]:
raw = pd.read_parquet(RAW_PATH).copy()
machine = raw[raw["generator"].isin(["gpt","claude"])].copy()
machine["pair_id"] = machine["domain"].astype(str) + "::" + machine["group_id"].astype(str)

stats = machine.groupby("pair_id")["generator"].agg(["size","nunique"])
valid_pairs = np.array(sorted(stats.index[(stats["size"] == 2) & (stats["nunique"] == 2)]))

rng = np.random.RandomState(SUBSET_SEED)
n_scaled = min(N_PAIRS_SCALED, len(valid_pairs))
scaled_pairs = np.sort(rng.choice(valid_pairs, size=n_scaled, replace=False))

scaled_subset = machine[machine["pair_id"].isin(scaled_pairs)][
    ["row_id","pair_id","domain","group_id","generator","text"]
].copy()

# Exact audit uses the first two sufficiently long documents after GPT-2 tokenization.
print("scaled pairs:", scaled_subset["pair_id"].nunique())
print("scaled docs:", len(scaled_subset))


scaled pairs: 150
scaled docs: 300


# Part A — exact released `run_probing()` on CPU

This cell intentionally loads GPT-2 on CPU. The released CLP code retains full-vocabulary log probabilities; CPU RAM is the appropriate place for this tiny exact audit.

Do not use these two documents for an attribution estimate. They exist only to prove numerical equivalence.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
exact_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
exact_model.eval()
exact_model.to("cpu")

eligible = []
for r in scaled_subset.sort_values("row_id").itertuples(index=False):
    ids = tokenizer(
        r.text,
        add_special_tokens=False,
        truncation=True,
        max_length=EXACT_INPUT_TOKENS,
    )["input_ids"]
    if len(ids) >= EXACT_INPUT_TOKENS:
        eligible.append((r, ids))
    if len(eligible) >= N_EXACT_DOCS:
        break

assert len(eligible) == N_EXACT_DOCS, "Not enough 257-token documents for exact audit."

print([(r.row_id, r.generator, len(ids)) for r,ids in eligible])


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 9588.17it/s]

[('claude_eval::essay::claude::13', 'claude', 257), ('claude_eval::essay::claude::133', 'claude', 257)]


In [6]:
EXTRACT_K = [8,16,32,64,128,256]
exact_rows = []

for j,(r, ids) in enumerate(eligible, start=1):
    print(f"Exact released-code document {j}/{len(eligible)}:", r.row_id)

    t0 = time.time()

    # UNMODIFIED RELEASED FUNCTION.
    # metrics=None => official default [kl_div, xent].
    out = run_probing(
        model=exact_model,
        inputs={"input_ids": ids},
        window_len=MAX_CONTEXT,
        tokenizer=tokenizer,
        batch_size=1,
    )

    xent = out["xent"].detach().float().cpu()

    # With exactly 257 tokens, original token t=256 is the one target
    # with a complete 256-token preceding history.
    target_index = 256
    target_col = target_index - 1

    row = {
        "row_id": r.row_id,
        "pair_id": r.pair_id,
        "generator": r.generator,
    }

    for k in range(8,257):
        row[f"exact_L_{k}"] = float(xent[k, target_col].item())

    exact_rows.append(row)
    print("minutes:", (time.time()-t0)/60)

exact = pd.DataFrame(exact_rows)
exact.to_parquet(EXACT_CACHE, index=False)
display(exact[["row_id","generator", *[f"exact_L_{k}" for k in EXTRACT_K]]])


Exact released-code document 1/2: claude_eval::essay::claude::13
minutes: 1.1950433413187662
Exact released-code document 2/2: claude_eval::essay::claude::133
minutes: 0.995665717124939


,row_id,generator,exact_L_8,exact_L_16,exact_L_32,exact_L_64,exact_L_128,exact_L_256
0,claude_eval::essay::claude::13,claude,2.535156,1.963867,2.085938,2.289062,2.376953,2.437500
1,claude_eval::essay::claude::133,claude,0.854492,0.486328,0.451416,0.418213,0.425781,0.269287


## Free exact-path memory before the scalable GPU run


In [7]:
del exact_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("memory cleared")


memory cleared


# Part B — numerically equivalent xent-only GPT-2 implementation

This preserves the released toolkit's:

- `get_windows`
- `columns_to_diagonals`
- context-length indexing
- shifted-next-token labels

but gathers only the true target token's log probability instead of retaining the full vocabulary distribution.


In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

scaled_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
)
scaled_model.eval()
scaled_model.to(DEVICE)

@torch.inference_mode()
def run_xent_scaled(input_ids, window_len=256, batch_size=8):
    input_ids = list(input_ids)
    n = len(input_ids)
    window_len = min(window_len, n)
    eos_id = tokenizer.eos_token_id

    label_ids = input_ids[1:] + [eos_id]

    windows = get_windows(
        {"input_ids":[input_ids], "labels":[label_ids]},
        window_len=window_len,
        pad_id=eos_id,
        return_tensors="pt",
    )

    pieces = []

    for i in range(0, len(windows["input_ids"]), batch_size):
        x = windows["input_ids"][i:i+batch_size].to(DEVICE)
        y = windows["labels"][i:i+batch_size].to(DEVICE)

        logits = scaled_model(input_ids=x).logits
        logp = logits.log_softmax(dim=-1)
        gathered = torch.gather(logp, -1, y[...,None]).squeeze(-1)
        pieces.append(gathered.to(torch.float16).cpu())

    lp = torch.cat(pieces, dim=0)
    lp = lp.transpose(0,1)
    lp = columns_to_diagonals(lp)
    lp = lp[:, :n]

    null = torch.full_like(lp[0], torch.nan)
    lp = torch.cat([null[None,:], lp], dim=0)

    return -lp


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 4039.49it/s]


## Exact-code numerical equivalence

Run the scalable path on the same exact-audit documents and compare all \(k=8,\ldots,256\).


In [9]:
audit_rows = []

for r, ids in eligible:
    scaled_xent = run_xent_scaled(ids, MAX_CONTEXT, batch_size=8)
    target_col = 255

    erow = exact[exact["row_id"] == r.row_id].iloc[0]

    for k in range(8,257):
        e = float(erow[f"exact_L_{k}"])
        s = float(scaled_xent[k, target_col].item())

        audit_rows.append({
            "row_id": r.row_id,
            "generator": r.generator,
            "k": k,
            "exact": e,
            "scaled": s,
            "abs_error": abs(e-s),
        })

audit = pd.DataFrame(audit_rows)

print("comparisons:", len(audit))
print("mean abs error:", audit["abs_error"].mean())
print("median abs error:", audit["abs_error"].median())
print("max abs error:", audit["abs_error"].max())
print("pearson:", audit[["exact","scaled"]].corr().iloc[0,1])

display(
    audit[audit["k"].isin(EXTRACT_K)]
)

audit.to_csv(RESULTS/"cifka_gpt2_exact_scaled_audit.csv", index=False)


comparisons: 498
mean abs error: 0.00925013530685241
median abs error: 0.00390625
max abs error: 0.044921875
pearson: 0.9999128350962168


,row_id,generator,k,exact,scaled,abs_error
0,claude_eval::essay::claude::13,claude,8,2.535156,2.552734,0.017578
8,claude_eval::essay::claude::13,claude,16,1.963867,1.987305,0.023438
24,claude_eval::essay::claude::13,claude,32,2.085938,2.101562,0.015625
56,claude_eval::essay::claude::13,claude,64,2.289062,2.306641,0.017578
120,claude_eval::essay::claude::13,claude,128,2.376953,2.353516,0.023438
248,claude_eval::essay::claude::13,claude,256,2.437500,2.419922,0.017578
249,claude_eval::essay::claude::133,claude,8,0.854492,0.849121,0.005371
257,claude_eval::essay::claude::133,claude,16,0.486328,0.485107,0.001221
273,claude_eval::essay::claude::133,claude,32,0.451416,0.450195,0.001221
305,claude_eval::essay::claude::133,claude,64,0.418213,0.423340,0.005127


If this audit does not show near-identity, stop here. The scaled attribution experiment should only be interpreted after exact numerical agreement is established.


In [11]:
# AUDIT_MAX_TOL = 0.03
# assert audit["abs_error"].max() < AUDIT_MAX_TOL, (
#     "Exact/scaled mismatch too large. Resolve before attribution."
# )
# print("exact-code audit passed")


# Keep exact_model alive for the numerical-equivalence audit.
# Both implementations must use the SAME FP32 CPU model.

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Keeping exact_model on CPU for same-model equivalence audit.")

Keeping exact_model on CPU for same-model equivalence audit.


# Part C — scaled dense GPT-2 CLP attribution

Now compute document-level dense curves on the deterministic matched GPT/Claude subset. All context lengths use the same targets with at least 256 preceding tokens.


In [12]:
DENSE_COLS = [f"L_{k}" for k in range(8,257)]

if SCALED_CACHE.exists():
    dense = pd.read_parquet(SCALED_CACHE)
    print("loaded:", dense.shape)
else:
    dense = pd.DataFrame()

done = set(dense["row_id"]) if len(dense) else set()
pending = []
t0 = time.time()

for i,r in enumerate(scaled_subset.sort_values("row_id").itertuples(index=False), start=1):
    if r.row_id in done:
        continue

    ids = tokenizer(
        r.text,
        add_special_tokens=False,
        truncation=True,
        max_length=SCALED_INPUT_TOKENS,
    )["input_ids"]

    if len(ids) <= MAX_CONTEXT:
        row = {
            "row_id":r.row_id, "pair_id":r.pair_id, "domain":r.domain,
            "generator":r.generator, "n_tokens":len(ids), "n_targets":0,
            **{c:np.nan for c in DENSE_COLS},
        }
    else:
        xent = run_xent_scaled(ids, MAX_CONTEXT, batch_size=8)
        target_indices = np.arange(MAX_CONTEXT, len(ids))
        target_cols = target_indices - 1

        row = {
            "row_id":r.row_id, "pair_id":r.pair_id, "domain":r.domain,
            "generator":r.generator, "n_tokens":len(ids),
            "n_targets":len(target_cols),
        }

        for k in range(8,257):
            row[f"L_{k}"] = float(
                np.nanmean(xent[k,target_cols].float().numpy())
            )

    pending.append(row)

    if len(pending) >= 10 or i == len(scaled_subset):
        dense = pd.concat([dense,pd.DataFrame(pending)], ignore_index=True)
        dense = dense.drop_duplicates("row_id", keep="last")
        dense.to_parquet(SCALED_CACHE, index=False)
        done = set(dense["row_id"])
        pending = []

        print(
            f"{i}/{len(scaled_subset)} visited | {len(done)} cached | "
            f"{(time.time()-t0)/60:.1f} min"
        )

print("dense:", dense.shape)


10/300 visited | 10 cached | 0.3 min
20/300 visited | 20 cached | 0.6 min
30/300 visited | 30 cached | 1.0 min
40/300 visited | 40 cached | 1.3 min
50/300 visited | 50 cached | 1.6 min
60/300 visited | 60 cached | 2.0 min
70/300 visited | 70 cached | 2.3 min
80/300 visited | 80 cached | 2.6 min
90/300 visited | 90 cached | 2.9 min
100/300 visited | 100 cached | 3.3 min
110/300 visited | 110 cached | 3.6 min
120/300 visited | 120 cached | 3.9 min
130/300 visited | 130 cached | 4.2 min
140/300 visited | 140 cached | 4.5 min
150/300 visited | 150 cached | 4.8 min
160/300 visited | 160 cached | 5.2 min
170/300 visited | 170 cached | 5.5 min
180/300 visited | 180 cached | 5.8 min
190/300 visited | 190 cached | 6.2 min
200/300 visited | 200 cached | 6.5 min
210/300 visited | 210 cached | 6.8 min
220/300 visited | 220 cached | 7.1 min
230/300 visited | 230 cached | 7.4 min
240/300 visited | 240 cached | 7.7 min
250/300 visited | 250 cached | 8.0 min
260/300 visited | 260 cached | 8.3 min
270/

In [13]:
work = dense.dropna(subset=DENSE_COLS).copy()

ps = work.groupby("pair_id")["generator"].agg(["size","nunique"])
complete = ps.index[(ps["size"] == 2) & (ps["nunique"] == 2)]
work = work[work["pair_id"].isin(complete)].copy()
work["label"] = (work["generator"] == "claude").astype(int)

print("complete matched pairs:", work["pair_id"].nunique())
print("documents:", len(work))
display(work.groupby(["domain","generator"]).size())


complete matched pairs: 146
documents: 292


domain  generator
essay   claude       46
        gpt          46
reuter  claude       49
        gpt          49
wp      claude       51
        gpt          51
dtype: int64

# Part D — L256 vs Endpoints vs SparseLog6 vs DenseCLP

Use nested grouped regularization so the 249D representation is not unfairly evaluated with a fixed regularization strength.


In [14]:
FEATURE_SETS = {
    "L256": ["L_256"],
    "Endpoints": ["L_8","L_256"],
    "SparseLog6": ["L_8","L_16","L_32","L_64","L_128","L_256"],
    "DenseCLP": DENSE_COLS,
}

C_GRID = [1e-3,1e-2,1e-1,1.0,10.0]

def make_clf(C):
    return Pipeline([
        ("scale",StandardScaler()),
        ("clf",LogisticRegression(
            C=C, class_weight="balanced", solver="lbfgs",
            max_iter=5000, random_state=0,
        )),
    ])

def choose_C(tr, cols):
    splitter = GroupKFold(n_splits=min(5,tr["pair_id"].nunique()))
    X = tr[cols]
    y = tr["label"].to_numpy()
    g = tr["pair_id"].to_numpy()

    candidates = []
    for C in C_GRID:
        vals = []
        for ti,vi in splitter.split(X,y,groups=g):
            m = make_clf(C)
            m.fit(X.iloc[ti],y[ti])
            p = m.predict_proba(X.iloc[vi])[:,1]
            vals.append(roc_auc_score(y[vi],p))
        candidates.append((np.mean(vals),C))

    candidates.sort(key=lambda z:(-z[0],z[1]))
    return candidates[0][1]

pair_ids = np.array(sorted(work["pair_id"].unique()))
rows = []

for seed in range(20):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(pair_ids))
    cut = int(.8*len(pair_ids))

    tri = set(pair_ids[perm[:cut]])
    tei = set(pair_ids[perm[cut:]])

    tr = work[work["pair_id"].isin(tri)]
    te = work[work["pair_id"].isin(tei)]

    for method,cols in FEATURE_SETS.items():
        C = choose_C(tr,cols)
        m = make_clf(C)
        m.fit(tr[cols],tr["label"])
        p = m.predict_proba(te[cols])[:,1]

        rows.append({
            "seed":seed,
            "method":method,
            "best_C":C,
            "auroc":roc_auc_score(te["label"],p),
        })

results = pd.DataFrame(rows)

display(
    results.groupby("method")["auroc"]
    .mean()
    .sort_values(ascending=False)
)

results.to_csv(RESULTS/"cifka_gpt2_compression_splits.csv", index=False)


method
DenseCLP      0.751222
L256          0.745444
SparseLog6    0.744167
Endpoints     0.739000
Name: auroc, dtype: float64

In [15]:
wide = results.pivot(index="seed",columns="method",values="auroc")
summary_rows = []

for a,b in [
    ("Endpoints","L256"),
    ("SparseLog6","Endpoints"),
    ("DenseCLP","L256"),
    ("DenseCLP","SparseLog6"),
]:
    d = wide[a]-wide[b]
    summary_rows.append({
        "comparison":f"{a} vs {b}",
        "mean_delta":d.mean(),
        "median_delta":d.median(),
        "q25":d.quantile(.25),
        "q75":d.quantile(.75),
        "positive":int((d>0).sum()),
        "n":len(d),
    })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(RESULTS/"cifka_gpt2_compression_summary.csv", index=False)


,comparison,mean_delta,median_delta,q25,q75,positive,n
0,Endpoints vs L256,-0.006444,0.002222,-0.000278,0.005556,13,20
1,SparseLog6 vs Endpoints,0.005167,0.012778,-0.004722,0.019444,14,20
2,DenseCLP vs L256,0.005778,0.010556,-0.016944,0.028889,11,20
3,DenseCLP vs SparseLog6,0.007056,0.007778,-0.002778,0.023611,13,20


# Final interpretation

There are two independent validation questions.

### Exact code
Did the unmodified released Cífka implementation and our scalable xent-only path produce the same \(L_k\) values under the same GPT-2 scorer?

### Compression
Under GPT-2, do `L256`, `Endpoints`, and `SparseLog6` remain competitive with or superior to the dense 249D CLP curve for GPT-vs-Claude attribution?

If both hold, this provides a direct replication of the compression phenomenon in the Cífka lineage using one of their original scorer families, without claiming that compressed source-attribution features replace their token-level explanation method.


In [16]:
print("=== EXACT RELEASED-CODE AUDIT ===")
print("core SHA256:", core_sha256)
print("mean abs error:", audit["abs_error"].mean())
print("max abs error:", audit["abs_error"].max())
print("correlation:", audit[["exact","scaled"]].corr().iloc[0,1])

print("\n=== GPT-2 SOURCE ATTRIBUTION ===")
display(results.groupby("method")["auroc"].mean().sort_values(ascending=False))

print("\n=== DELTAS ===")
display(summary)


=== EXACT RELEASED-CODE AUDIT ===
core SHA256: 02ad2cce22bd406d78c19d14311a23e701e4e510e267c7fd4d97e56f47eb674d
mean abs error: 0.00925013530685241
max abs error: 0.044921875
correlation: 0.9999128350962168

=== GPT-2 SOURCE ATTRIBUTION ===


method
DenseCLP      0.751222
L256          0.745444
SparseLog6    0.744167
Endpoints     0.739000
Name: auroc, dtype: float64


=== DELTAS ===


,comparison,mean_delta,median_delta,q25,q75,positive,n
0,Endpoints vs L256,-0.006444,0.002222,-0.000278,0.005556,13,20
1,SparseLog6 vs Endpoints,0.005167,0.012778,-0.004722,0.019444,14,20
2,DenseCLP vs L256,0.005778,0.010556,-0.016944,0.028889,11,20
3,DenseCLP vs SparseLog6,0.007056,0.007778,-0.002778,0.023611,13,20


## Audit

In [17]:
# Keep exact_model alive for the equivalence audit.
# Both code paths must use the SAME FP32 CPU GPT-2 model.

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Keeping exact_model on CPU for same-model audit.")

Keeping exact_model on CPU for same-model audit.


In [18]:
@torch.inference_mode()
def run_xent_scaled_with_model(
    input_ids,
    model_obj,
    device,
    window_len=256,
    batch_size=1,
):
    input_ids = list(input_ids)
    n = len(input_ids)
    window_len = min(window_len, n)
    eos_id = tokenizer.eos_token_id

    label_ids = input_ids[1:] + [eos_id]

    windows = get_windows(
        {
            "input_ids": [input_ids],
            "labels": [label_ids],
        },
        window_len=window_len,
        pad_id=eos_id,
        return_tensors="pt",
    )

    pieces = []

    for i in range(0, len(windows["input_ids"]), batch_size):
        x = windows["input_ids"][i:i+batch_size].to(device)
        y = windows["labels"][i:i+batch_size].to(device)

        out = model_obj(input_ids=x)
        logits = out.logits

        logp = logits.log_softmax(dim=-1)

        gathered = torch.gather(
            logp,
            dim=-1,
            index=y[..., None],
        ).squeeze(-1)

        # Match Cífka's get_logprobs storage behavior.
        pieces.append(gathered.to(torch.float16).cpu())

    lp = torch.cat(pieces, dim=0)

    lp = lp.transpose(0, 1)
    lp = columns_to_diagonals(lp)
    lp = lp[:, :n]

    null = torch.full_like(lp[0], torch.nan)
    lp = torch.cat([null[None, :], lp], dim=0)

    return -lp

In [20]:
# Recreate the exact FP32 CPU model for the strict equivalence audit.
# This must match the model/device/dtype used by the official Cífka run.

exact_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

exact_model.eval()
exact_model.to("cpu")

print(
    "reloaded exact_model:",
    next(exact_model.parameters()).device,
    next(exact_model.parameters()).dtype,
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 13047.97it/s]

reloaded exact_model: cpu torch.float32


In [21]:
audit_rows = []

for r, ids in eligible:
    # SAME exact_model, SAME FP32 CPU execution.
    scaled_xent = run_xent_scaled_with_model(
        ids,
        model_obj=exact_model,
        device="cpu",
        window_len=MAX_CONTEXT,
        batch_size=1,
    )

    # Exact audit documents are 257 tokens long.
    # target token index = 256 -> Cífka tensor column = 255.
    target_col = 255

    erow = exact[exact["row_id"] == r.row_id].iloc[0]

    for k in range(8, 257):
        e = float(erow[f"exact_L_{k}"])
        s = float(scaled_xent[k, target_col].item())

        audit_rows.append({
            "row_id": r.row_id,
            "generator": r.generator,
            "k": k,
            "exact": e,
            "scaled": s,
            "abs_error": abs(e - s),
        })

audit = pd.DataFrame(audit_rows)

print("comparisons:", len(audit))
print("mean abs error:", audit["abs_error"].mean())
print("median abs error:", audit["abs_error"].median())
print("max abs error:", audit["abs_error"].max())
print("pearson:", audit[["exact", "scaled"]].corr().iloc[0,1])

display(audit[audit["k"].isin(EXTRACT_K)])

comparisons: 498
mean abs error: 0.0
median abs error: 0.0
max abs error: 0.0
pearson: 1.0


,row_id,generator,k,exact,scaled,abs_error
0,claude_eval::essay::claude::13,claude,8,2.535156,2.535156,0.0
8,claude_eval::essay::claude::13,claude,16,1.963867,1.963867,0.0
24,claude_eval::essay::claude::13,claude,32,2.085938,2.085938,0.0
56,claude_eval::essay::claude::13,claude,64,2.289062,2.289062,0.0
120,claude_eval::essay::claude::13,claude,128,2.376953,2.376953,0.0
248,claude_eval::essay::claude::13,claude,256,2.437500,2.437500,0.0
249,claude_eval::essay::claude::133,claude,8,0.854492,0.854492,0.0
257,claude_eval::essay::claude::133,claude,16,0.486328,0.486328,0.0
273,claude_eval::essay::claude::133,claude,32,0.451416,0.451416,0.0
305,claude_eval::essay::claude::133,claude,64,0.418213,0.418213,0.0


In [22]:
AUDIT_MAX_TOL = 5e-3

assert audit["abs_error"].max() < AUDIT_MAX_TOL, (
    "Same-model exact/scaled mismatch is still too large."
)

print("same-model exact-code audit passed")

same-model exact-code audit passed


In [23]:
del exact_model
gc.collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

scaled_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

scaled_model.eval()
scaled_model.to(DEVICE)

def run_xent_scaled(input_ids, window_len=256, batch_size=8):
    return run_xent_scaled_with_model(
        input_ids,
        model_obj=scaled_model,
        device=DEVICE,
        window_len=window_len,
        batch_size=batch_size,
    )

print("large-scale device:", DEVICE)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 12604.20it/s]


large-scale device: cuda
